In [ ]:
import openai
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import time
import math
import os
import csv
from dotenv import load_dotenv

# ==============================================================================
# 1. 설정 정보 (사용자 수정 필요)
# ==============================================================================

# ⚠️ 본인의 실제 OpenAI API 키를 입력하세요 (이전 키는 반드시 폐기하세요!)
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ⚠️ 구글 서비스 계정 JSON 파일 경로
SERVICE_ACCOUNT_FILE = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# ⚠️ 작업할 구글 스프레드시트의 URL
SPREADSHEET_URL = os.getenv("SPREADSHEET_URL")

missing = [name for name, value in {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "GOOGLE_APPLICATION_CREDENTIALS": SERVICE_ACCOUNT_FILE,
    "SPREADSHEET_URL": SPREADSHEET_URL,
}.items() if not value]
if missing:
    raise RuntimeError(f".env에 다음 값을 설정해 주세요: {', '.join(missing)}")

# ⚠️ 분석할 워크시트 이름들
TARGET_SHEETS = ["리더십이미지_구성원"]  

# ✅ PPT 슬라이드 분할 기준 설정
MAX_CHARS_PER_CELL = 800  
MAX_LINES_PER_CELL = 10    
GROUPING_THRESHOLD = 5     

# ✅ 로컬 백업 파일 설정 (주피터랩 중간 멈춤 방지용)
BACKUP_FOLDER = "./backup_results"
os.makedirs(BACKUP_FOLDER, exist_ok=True)

# ==============================================================================
# 2. 프롬프트 정의
# ==============================================================================
# [1+2단계 통합] 핵심 문장 추출 및 줄바꿈 정규화
PROMPT_MAIN = """다음 [입력 텍스트]에서 핵심 내용을 추출하여 정규화 및 요약해주세요.

[지시사항]
1. 각 핵심 내용은 반드시 'ㆍ ' 기호로 시작해야 합니다.
2. 여러 항목이 추출될 경우, 반드시 각 항목마다 '줄바꿈(Enter)'을 하여 보기 좋게 정리해주세요.
3. 결과 외에 어떠한 안내 문구(예: "정리된 내용은 다음과 같습니다" 등)도 절대 포함하지 마세요.

# 문장 추출 및 정규화 예제
## case 1
### input
없음. 옛날 직장인?
### output
ㆍ 옛날 직장인

## case 2
### input
고 0 좋은사람
### output
ㆍ 고
ㆍ 좋은사람

## case 3
### input
- 추진력이 아주 강함 (일로 만들어냄) - 협상력 뛰어남 (우리의/협력사의 니즈와 페인포인트를 꿰뚫어 협의를 이끌어냄) - 솔선수범함 (제일 많음은 기본이고, 마다하지 않고 행동함) - 문제를 해결함 (캐치해내고, 현실화함) - 팀원들의 역량 계발을 적극 지원함 (업무 순환에 대해 고민함, 배경과 WHY를 항상 강조함)전문성, 행동력, 업무적 care, 헌신0
### output
ㆍ 추진력이 아주 강함(일로 만들어냄). 협상력 뛰어남 (우리의, 협력사의 니즈와 페인포인트를 꿰뚫어 협의를 이끌어냄), 솔선수범(제일 많음은 기본이고, 마다하지 않고 행동함), 문제를 해결(캐치해내고, 현실화), 팀원들의 역량 계발을 적극 지원(업무 순환에 대해 고민함. 배경과 WHY를 항상 강조)
ㆍ 전문성, 행동력, 업무적 care, 헌신

## case 4
### input
솔순수범
### output
ㆍ 솔선수범

## case 5
### input
구성원간의 송통
### output
ㆍ 구성원 간의 소통

## case 6
### input
- 끊임없는 고민 및 전략적 Approach - 팀원들에게 전달하지 않는 점
### output
ㆍ 끊임없는 고민 및 전략적 Approach. 팀원들에게 전달하지 않는 점

## case 7
### input
실무진과 밀접하게 업무를 진행하고, 이슈에 대해 신속하게 판단/대응함  팀원들의 업무 등에 잘 할 수 있도록 많은 관심을 가져주는 것- 상황/능력을 잘 고려하여 목표를 달성 - 솔선수범 하는 리더 - 다독이는 능력이 뛰어남
### output
ㆍ 실무진과 밀접하게 업무를 진행하고, 이슈에 대해 신속하게 판단, 대응
ㆍ 팀원들의 업무 등에 잘 할 수 있도록 많은 관심을 가져주는 것
ㆍ 상황, 능력을 잘 고려하여 목표를 달성. 솔선수범하는 리더. 다독이는 능력이 뛰어남

## case 8
### input
긍정적인 마인드추진력추진력강한 추진력과 전략적 사고
### output
ㆍ 긍정적인 마인드
ㆍ 추진력
ㆍ 추진력
ㆍ 강한 추진력과 전략적 사고

## case 9
### input
. 솔선수범  현장감을 가지고 솔선수범 모름 솔선수범 부하직원에 대한 온유한 태도 차분한 성격을 가졌음 구성원과의 소통 및 존중 조직 단합력 솔선수범 빠른 일처리 직원간 소통 솔선수범   솔선수범 솔선수범 소통 추진력 차분하면서 경청하고 핵심을 찌르는 포인트. 소통,친근하고 건위의식이 없습니다. 리더쉽 편안한 소통 상대방 읭견을 경청 실행력 푸근한 인상의 친근함 일 처리를 뒤로 미루지 않고 깔금하게한다 생각을 강요하지 않고 존중하며 이클어줌 솔선수범 성실함 오픈마인드와 이해심 다양한경험과 변화하려는 의지 . 논리적인 생각 디테일 친화력 특별한 사항은 없다 . 0 짜증내지 않고 부드럽게 풀어가는 방식 0 구성원 존중과 배려
### output
ㆍ 솔선수범
ㆍ 현장감을 가지고 솔선수범
ㆍ 솔선수범
ㆍ 부하직원에 대한 온유한 태도
ㆍ 차분한 성격을 가졌음
ㆍ 구성원과의 소통 및 존중
ㆍ 조직 단합력
ㆍ 솔선수범
ㆍ 빠른 일처리
ㆍ 직원간 소통
ㆍ 솔선수범
ㆍ 솔선수범
ㆍ 솔선수범
ㆍ 소통
ㆍ 추진력
ㆍ 차분하면서 경청하고 핵심을 찌르는 포인트
ㆍ 소통. 친근하고 권위의식이 없음
ㆍ 리더쉽
ㆍ 편안한 소통
ㆍ 상대방 의견을 경청
ㆍ 실행력
ㆍ 푸근한 인상의 친근함
ㆍ 일 처리를 뒤로 미루지 않고 깔끔하게 함
ㆍ 생각을 강요하지 않고 존중하며 이끌어 줌
ㆍ 솔선수범
ㆍ 성실함
ㆍ 오픈마인드와 이해심
ㆍ 다양한경험과 변화하려는 의지
ㆍ 논리적인 생각
ㆍ 디테일
ㆍ 친화력
ㆍ 짜증내지 않고 부드럽게 풀어가는 방식
ㆍ 구성원 존중과 배려

## case 10
### input
특별히 개선이 필요한 부분은 없음
### output
ㆍ 응답 내용 없음

## case 11
### input
특별히 없습니다. 활성화할 필요 있음 (회의, 공유 등) R&R을 이야기할 필요 있음. 위임할 필요가 있음. 없음휴식이 필요함업무 분배
### output
ㆍ 활성화할 필요 있음(회의, 공유 등). R&R을 이야기할 필요 있음. 위임할 필요가 있음
ㆍ 휴식이 필요
ㆍ 업무 분배

## case 12
### input
결정력 추진력 판단력 이해력 기타 등등 0 없음 안건에 대한 빠른 결정이 필요. 1. 좀 느린 피드백 2. 업무 지정이 필요
### output
ㆍ 결정력 추진력 판단력 이해력 기타 등등
ㆍ 안건에 대한 빠른 결정이 필요
ㆍ 좀 느린 피드백. 업무 지정이 필요

## case 13
### input
직책자 경험/경력 .스트레스를 많이 받음. 스트레스 조절이 필요해 보임 . 0
### output
ㆍ 직책자 경험, 경력
ㆍ 스트레스를 많이 받음. 스트레스 조절이 필요해 보임

## case 14
### input
개인 역량을 위한 제안 1 없음 없음반영되는 것 같습니다. 좀 더 밀어붙이셨으면 좋겠습니다. 크게 없는거 같다
### output
ㆍ 개인 역량을 위한 제안
ㆍ 반영되는 것 같음. 좀 더 밀어붙이셨으면 좋겠음

## case 15
### input
지금처럼 든든한 지원군이 되어주세요! 만족합니다. 화..이팅
### output
ㆍ 지금처럼 든든한 지원군이 되어 주시기 바람
ㆍ 만족
ㆍ 화이팅

## case 16
### input
실행주의 / 빠른 피드백 / 넓은 시
### output
ㆍ 실행주의, 빠른 피드백, 넓은 시야

## case 17
### input
일찍 일어난 새가 먼저 벌레를 먹는다. 모든 사람과 두루 관계가 좋은 유재석.1. 타 팀리더와 다르게 군대의 행보관과 같이 모든 업무에 다 괌심을 주시고 솔선수범하여, 구성원의 업무상 어려운 부분이 잘 해결될 수 있도록 도움을 주십니다. 2. 권위적이지 않아 업무적, 사적인 내용 모두 편안히 대화할 수 있어 친구와 대화하듯이 이야기 할 수 있어 편하고 좋습니다. .보살 소통의 리더십긍정, 온유 매사 긍정적이고 성격과 태도가 온화하고 부드럽다.KTX 독수리짜증 .12색상 색연필 - 개성있게 표현하고, 상황에 따라 맞는 행동을 함 백과사전 - 정보와 지식이 모두 있는 장소대쪽같은/닫혀있는나를 따르라/나폴레옹바다 위의 무모한 항해자-나침반도, 세심한 계획도 없이 항해를 지시함 사막의 건축가-계획을 가지고 있지만, 그리는 설계도만 제시함- 프로페셔널을 전파하는 코치: 커뮤니케이션 방법을 바탕으로 피드백을 전해주는 코치라고 생각합니다. 진심어린 피드백을 전달해 주십니다. - 나침반과 지도: 조직과 구성원의 지도를 그려주신 후, 역할과 책임, 적절한 나침반(피드백)을 제공해 주시는 리더십입니다. 구성원이 성장할 수 있도록 해주십니다. 구성원을 믿어준다. 새로운 아이디어를 자주 낸다.- 전략기획팀 팀장으로서, 업무를 추진/위임/배분하여 주십니다. 나아가, 의미를 발견하시어, 리더십을 갖추셨다고 생각합니다.  -  개방적으로 수용하고, 공유 문화를 촉진하며, 리더십을 갖추셨다고 생각합니다.   - 솔선수범하여 보여주시며, 리더십을 갖추셨다고 생각합니다.담배직접적으로 표현을 좀 더 하셨으면 합니다. 약간의 강박관념을 덜어내셨으면 합니다.1. 그룹장님도 야근을 많이 안 하셨으면 좋겠습니다.  2. 없습니다. 그룹장님 본인께서 야근을 많이 하시는 것 같습니다. 구성원들에게 맡기고 퇴근해주시면 좋을 것 같습니다.본인 워라밸 및 건강관리 外 N/A업무가 상대적으로 여유있는 구성원들의 역량을 끌어올릴수 있게 더 힘써 주시면 좋겠다.개인의 스트레스 1. SK환경과학기술원 대표 팀 2. 역사가 깊은 팀 Module 소자 개발 담당 Module 개발의 최종 담당 등대같은 역할로 방향성과 목적지를 알려 준다. 1. 회사를 지키는 수호신 : 우리 팀은 가장 먼저, 가장 많이 외부의 압박을 받게 된다. 그걸 온몸으로 막고 버텨내야 한다. 그만큼 업무 강도가 높지만, 구성원 모두가 애사심과 자부심으로 똘똘 뭉쳐 성과를 창출한다. 2. 5분 대기조 : 우리 팀은 외부 SH와의 갑을관계에서 철저한 을이다. 그렇기에 조금 과장하자면 1년 365일 하루 24시간 내내 긴장의 끈을 놓을 수 없는 상태이다. 말 한마디가 조심스럽고 민감하다. 늘 어렵고 힘든 상황에서 근무하고 있다. 쓰레기통 층층시하 .특수부대: 그룹 차원에서 중대한 사안이 생길 때, 우리 조직이 먼저 투입되어 신속대응을 합니다. 하이에나: 초원에서 먹을 것을 찾아 마지막 고기 한점까지 발라먹는 하이에나처럼, 우리 조직도 마지막 한 치의 오차도 없이 업무를 처리합니다. 보수적 제한적 의사결정 재무 자율성 수평적이고 평등한 가족 같은 팀입니다 융합 , 각자의 전문성을 따로 또 같이 조화롭게 조율하고 경험과 센스를 더하여 좋은 결과물을 만들어낸다 사이가 좋고 서로 가깝게 지낸다 과제, 업무 진행시 도전적이고 어떤식으로도 일을 완수해내려는 노력을 많이 한다 각자 다른 업무를 보다가도 한 가지 이슈사항에 대해 모일 줄 아는 단합력이 강점입니다. 그 중심에는 중심을 잡아주시는 PL 창님들이 계십니다. 수평적인 분위기가 강점이라고 생각됩니다.공장장님 직속 조직으로 서산 3동 공장의 성공적인 증설이라는 명확한 목적으로 구성된 조직으로 공통된 목표를 갖고 효과적이고 효율적인 업무 진행이 가능하다고 생각합니다. 납기를 너무 잘 맞춰서 일정에 여유를 가졌으면 합니다.
### output
ㆍ 일찍 일어난 새가 먼저 벌레를 먹음. 모든 사람과 두루 관계가 좋은 유재석
ㆍ 타 팀 리더와 다르게 군대의 행보관과 같이 모든 업무에 다 관심을 주시고 솔선수범하여, 구성원의 업무상 어려운 부분이 잘 해결될 수 있도록 도움을 주심. 권위적이지 않아 업무적, 사적인 내용 모두 편안히 대화할 수 있어 친구와 대화하듯이 이야기 할 수 있어 편하고 좋음
ㆍ 보살, 소통의 리더십
ㆍ 긍정, 온유, 매사 긍정적이고 성격과 태도가 온화하고 부드러움
ㆍ KTX, 독수리
ㆍ 짜증 
ㆍ 12색상 색연필: 개성있게 표현하고, 상황에 따라 맞는 행동을 함. 백과사전: 정보와 지식이 모두 있는 장소
ㆍ 대쪽 같은, 닫혀 있는
ㆍ 나를 따르라, 나폴레옹
ㆍ 바다 위의 무모한 항해자: 나침반도, 세심한 계획도 없이 항해를 지시. 사막의 건축가: 계획을 가지고 있지만, 그리는 설계도만 제시
ㆍ - 프로페셔널을 전파하는 코치: 커뮤니케이션 방법을 바탕으로 피드백을 전해주는 코치라고 생각. 진심 어린 피드백을 전달해 주심. 나침반과 지도: 조직과 구성원의 지도를 그려 주신 후, 역할과 책임, 적절한 나침반(피드백)을 제공해 주시는 리더십. 구성원이 성장할 수 있도록 해주심
ㆍ 구성원을 믿어줌. 새로운 아이디어를 자주 냄
ㆍ 전략기획팀 팀장으로서, 업무를 추진/위임/배분하여 주심. 나아가, 의미를 발견하시어, 리더십을 갖추셨다고 생각. 개방적으로 수용하고, 공유 문화를 촉진하며, 리더십을 갖추셨다고 생각. 솔선수범하여 보여주시며, 리더십을 갖추셨다고 생각
ㆍ 담배
ㆍ 직접적으로 표현을 좀 더 하셨으면 함. 약간의 강박관념을 덜어내셨으면 함
ㆍ 그룹장님도 야근을 많이 안 하셨으면 좋겠음
ㆍ 그룹장님 본인께서 야근을 많이 하시는 것 같음. 구성원들에게 맡기고 퇴근해주시면 좋을 것 같음
ㆍ 본인 워라밸 및 건강관리
ㆍ 업무가 상대적으로 여유 있는 구성원들의 역량을 끌어올릴 수 있게 더 힘써 주시면 좋겠음
ㆍ 개인의 스트레스 
ㆍ SK 환경과학기술원 대표 팀, 역사가 깊은 팀
ㆍ Module 소자 개발 담당, Module 개발의 최종 담당
ㆍ 등대 같은 역할로 방향성과 목적지를 알려 줌
ㆍ 회사를 지키는 수호신: 우리 팀은 가장 먼저, 가장 많이 외부의 압박을 받게 됨. 그걸 온몸으로 막고 버텨내야 함. 그만큼 업무 강도가 높지만, 구성원 모두가 애사심과 자부심으로 똘똘 뭉쳐 성과를 창출. 5분 대기조: 우리 팀은 외부 SH와의 갑을관계에서 철저한 을. 그렇기에 조금 과장하자면 1년 365일 하루 24시간 내내 긴장의 끈을 놓을 수 없는 상태. 말 한마디가 조심스럽고 민감. 늘 어렵고 힘든 상황에서 근무
ㆍ 쓰레기통
ㆍ 층층시하
ㆍ 특수부대: 그룹 차원에서 중대한 사안이 생길 때, 우리 조직이 먼저 투입되어 신속대응. 하이에나: 초원에서 먹을 것을 찾아 마지막 고기 한점까지 발라 먹는 하이에나처럼, 우리 조직도 마지막 한 치의 오차도 없이 업무를 처리
ㆍ 보수적, 제한적, 의사결정
ㆍ 재무, 자율성
ㆍ 수평적이고 평등한 가족 같은 팀
ㆍ 융합 , 각자의 전문성을 따로 또 같이 조화롭게 조율하고 경험과 센스를 더하여 좋은 결과물을 만들어낸다 사이가 좋고 서로 가깝게 지냄
ㆍ 과제, 업무 진행시 도전적이고 어떤식으로도 일을 완수해내려는 노력을 많이 함
ㆍ 각자 다른 업무를 보다가도 한 가지 이슈사항에 대해 모일 줄 아는 단합력이 강점. 그 중심에는 중심을 잡아주시는 PL님들이 계심
ㆍ 수평적인 분위기가 강점이라고 생각
ㆍ 공장장님 직속 조직으로 서산 3동 공장의 성공적인 증설이라는 명확한 목적으로 구성된 조직으로 공통된 목표를 갖고 효과적이고 효율적인 업무 진행이 가능하다고 생각
ㆍ 납기를 너무 잘 맞춰서 일정에 여유를 가졌으면 함

## case 18
### input
특이사항없음.중장기 계획이 없이, 단기(1년) 계획만 있다고객사의 이익과 우리회사의 이익이 달라서 상호충돌한다.서로의 개성이 워낙 다양하다 보니 업무 외 구심점을 가질 수 있는 주재를 찾기가 쉽지 앖습니다. 우리 조직은 매년 동일한 패턴의 경영목표(매출/마진 등), 경영전략으로 업무를 수행해왔습니다. 목표와 방향성이 바뀌어야 한다고 생각합니다.맡은 업무의 본질(아이티 기술력)에 대해 보완 해야 한다.팀별 규모가 너무 크다. 세분화할 필요가 있음고객사가 추구하는 바가 시스템의 안정적인 운영에 초점이 맞추어 있다보니 새로운 / 과감한 시도가 적고 이를 지원하는데 있어 부족함이 많은 듯 합니다. 이에 조직이 점점 더 패시브하게 바뀌어가오 있다고 느껴집니다. 좀 더 능동적인 조직으로 탈바꿈할 수 있도록 변화가 필요하다고 생각됩니다.IT 역량 조직 목표을 위한 리더 - 구성원간 다양한 소통 채널, 소통 기회, 소통 횟수, 소통 피드백이 확대 되었으면 합니다전문가들이 많으니 밀어부치는 AGS 대한 우려가 매우 높습니다. 진행에 있어 구성원의 의견도 청취해 주세요.S3 TF는 여러 조직의 구성원이 모인 자리 입니다. 무조건 적인 CAPEX 절감이 아닌 기술 출신 구성원의 의견이 우선적으로 반영되었으면 좋겠습니다.외부 요인으로 진행중이던 프로젝트가 무산이 되는 경우가 있습니다. 이런 경우에는 팀, 그리고 개인의 성과 및 성취감과 직결되므로 미리 이런 장애요인들을 파악하고, 사전에 방지하는 툴들을 마련해두아야 합니다.회식/팀 행사 때 사용하는 지원 비용이 부족합니다.
### output
ㆍ 중장기 계획이 없이, 단기(1년) 계획만 있음
ㆍ 고객사의 이익과 우리회사의 이익이 달라서 상호충돌
ㆍ 서로의 개성이 워낙 다양하다 보니 업무 외 구심점을 가질 수 있는 주재를 찾기가 쉽지 않음
ㆍ 우리 조직은 매년 동일한 패턴의 경영목표(매출/마진 등), 경영전략으로 업무를 수행해왔습니다. 목표와 방향성이 바뀌어야 한다고 생각
ㆍ 맡은 업무의 본질(아이티 기술력)에 대해 보완해야 함
ㆍ 팀별 규모가 너무 큼. 세분화할 필요가 있음
ㆍ 고객사가 추구하는 바가 시스템의 안정적인 운영에 초점이 맞추어 있다 보니 새로운, 과감한 시도가 적고 이를 지원하는데 있어 부족함이 많은 듯함. 이에 조직이 점점 더 패시브하게 바뀌어 가고 있다고 느껴 짐. 좀 더 능동적인 조직으로 탈바꿈할 수 있도록 변화가 필요하다고 생각
ㆍ IT 역량 조직 목표를 위한 리더: 구성원간 다양한 소통 채널, 소통 기회, 소통 횟수, 소통 피드백이 확대되었으면 함
ㆍ 전문가들이 많으니 밀어 부치는 AGS 대한 우려가 매우 높음. 진행에 있어 구성원의 의견도 청취해 주시기 바람
ㆍ S3 TF는 여러 조직의 구성원이 모인 자리. 무조건 적인 CAPEX 절감이 아닌 기술 출신 구성원의 의견이 우선적으로 반영되었으면 좋겠음
ㆍ 외부 요인으로 진행 중이던 프로젝트가 무산이 되는 경우가 있음. 이런 경우에는 팀, 그리고 개인의 성과 및 성취감과 직결되므로 미리 이런 장애 요인들을 파악하고, 사전에 방지하는 툴 들을 마련 해 두어야 함
ㆍ 회식, 팀 행사 때 사용하는 지원 비용이 부족

## case 19
### input
- 워라밸 - 보상-
### output
ㆍ 워라밸, 보상

## case 20
### input
아직은 잘 모르겠음   - . 감사합니다. 구성원 행복을 추진하기에 팀운영비 부족. 담당님 보고서구체적인 사례가 있을 정도는 아니지만 중요한 일이 있을 때 구성원의 압박과 소모가 심하기 때문에 당면한 과제가 없는 구성원의 경우 좀 적극적으로 쉬어서 여력을 회복하는 형식을 좀 더 장려했으면 좋겠습니다.우리팀 100%로 끌고나갈 수가 없는 업무들입니다. 우리팀의 성과를 결정하는 상황이 많습니다. 국내외 입찰 등의 경우입니다. 설계성과품을 완성하지만, 경쟁사 대비 고품질, 상위의 성과품을 완성하는 것과는 별개로 국내입찰의 성패는 국내영업의 성패에 달려있고, 해외 입찰의 경우 해외견적에 달려있습니다. 구성원 행복도 모두 놓치는 상황이 발생했던 것 같습니다.과거 조직에서의 사례를 떠올려보면, 1) 리더의 업무 방향성 미제시, 구성원 방치 2) 관계 중심의 편향된 의사결정. 의사결정 과정에서의 불투명함. 특정 가까운 사람 의견 중심의 의사결정 
### output
ㆍ 구성원 행복을 추진하기에 팀 운영비 부족
ㆍ 담당님 보고서
ㆍ 구체적인 사례가 있을 정도는 아니지만 중요한 일이 있을 때 구성원의 압박과 소모가 심하기 때문에 당면한 과제가 없는 구성원의 경우 좀 적극적으로 쉬어서 여력을 회복하는 형식을 좀 더 장려했으면 좋겠음
ㆍ 우리팀 100%로 끌고나갈 수가 없는 업무들. 우리팀의 성과를 결정하는 상황이 많음. 국내외 입찰 등의 경우임. 설계성과품을 완성하지만, 경쟁사 대비 고품질, 상위의 성과품을 완성하는 것과는 별개로 국내입찰의 성패는 국내영업의 성패에 달려있고, 해외 입찰의 경우 해외견적에 달려있음. 구성원 행복도 모두 놓치는 상황이 발생했던 것 같음
ㆍ 과거 조직에서의 사례를 떠올려보면, 리더의 업무 방향성 미제시, 구성원 방치, 관계 중심의 편향된 의사결정,  의사결정 과정에서의 불투명함. 특정 가까운 사람 의견 중심의 의사결정 

## case 21
### input
잘 모르겠음팀장 임명되신 후 이제 1달이라 아직 모르겠습니다!가끔 말걸이 어려운 포스를 보여주심없음
### output
ㆍ 가끔 말걸이 어려운 포스를 보여주심

## case 22
### input
소통없습니다.없음.X없음
### output
ㆍ 소통

## case 23
### input
 잘 모르겠습니다.  Newton2 화이팅입니다. 건강 지금도 너무 좋습니다. 올 한해 프로젝트를 잘 이끌어주세요~ 팀이 더 잘될 수 있도록 리더 스스로의 원동력이 떨어지지 않았으면 좋겠습니다. 지금처럼 해주시면 될 것 같습니다 딱히 떠오르는게 없음특이사항 없음
### output
ㆍ Newton2 화이팅
ㆍ 건강
ㆍ 지금도 너무 좋음
ㆍ 올 한해 프로젝트를 잘 이끌어주시기 바람
ㆍ 팀이 더 잘될 수 있도록 리더 스스로의 원동력이 떨어지지 않았으면 좋겠음
ㆍ 지금처럼 해주시면 될 것 같음

## case 24
### input
없음 0
### output
ㆍ 응답 내용 없음

## case 25
### input
마땅히 없습니다 구성원 역량 확보를 위한 교육지원 0 없음, 없음 의사 결정 시 신속성  없습니다. . 없음 23, 24번 답변과 동일하게 대면할 일이 적어 모르겠습니다.
### output
ㆍ 구성원 역량 확보를 위한 교육 지원
ㆍ 의사 결정 시 신속성

## case 26
### input
(1) 풍부한 품질 업무 경험을 바탕으로 어떤 문제에 대한 논리적 접근 시야가 넓고, 문제 해결 능력이 탁월하다. (2) 팀원들과 격 없이 융합하고 이끌어가는 능력이 탁월하다.
### output
ㆍ 풍부한 품질 업무 경험을 바탕으로 어떤 문제에 대한 논리적 접근 시야가 넓고, 문제 해결 능력이 탁월함. 팀원들과 격 없이 융합하고 이끌어가는 능력이 탁월함

## case 27
### input
 NA
### output
ㆍ 응답 내용 없음 

[입력 텍스트]
{text}
### output
"""

# [3단계] 주제별 그룹화
PROMPT_GROUPING = """다음 [입력 텍스트]를 분석하여 유사한 주제끼리 묶어주세요.
각 주제별로 적절한 타이틀(키워드)을 달고, 아래 형식에 맞춰 작성해주세요.
내용은 입력된 문장의 그대로 유지하며 나열해주세요.

**주의사항:**
1. 각 줄 사이에는 빈 줄(엔터)을 넣지 말고, 바로 다음 줄에 내용을 이어 작성해주세요.
2. 각 줄의 시작은 'ㆍ '로 시작해주세요.
3. 출력 결과에 이 지시사항이나 안내 문구는 절대 포함하지 마세요.

형식:
ㆍ [주제] : 내용1; 내용2

예시:
ㆍ [소통] : 위클리 뿐만 아니라, 직책자에게 공유 되는 내용 중 구성원이 알아야하는 내용에 대해서는 반드시 공유; 격의 없는 소통을 위해 서로 영어 이름을 부름
ㆍ [업무 효율] : 불필요한 회의를 줄이고 핵심 안건 위주로 짧게 진행함

[입력 텍스트]
{text}
"""


# ==============================================================================
# 3. 핵심 로직
# ==============================================================================

# 클라이언트 초기화
client = openai.OpenAI(api_key=OPENAI_API_KEY)

def authenticate_gspread(sheet_url, sheet_name):
    try:
        scope = ['https://spreadsheets.google.com/feeds', 'https://www.googleapis.com/auth/drive']
        creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scope)
        gc = gspread.authorize(creds)
        sh = gc.open_by_url(sheet_url)
        return sh.worksheet(sheet_name)
    except Exception as e:
        print(f"❌ 인증 또는 시트 '{sheet_name}' 로딩 실패: {e}")
        return None

def ensure_columns(worksheet, required_cols):
    if worksheet.col_count < required_cols:
        worksheet.add_cols(required_cols - worksheet.col_count)

def call_gpt(text, prompt_template, model="gpt-5.4"): # 💡 속도/가성비 최고인 gpt-4o-mini 적용
    if pd.isna(text) or str(text).strip() == "":
        return "응답 내용 없음"

    prompt = prompt_template.format(text=text)

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"⚠️ GPT 호출 실패 (시도 {attempt+1}): {e}")
            time.sleep(2 ** attempt)
            
    return f"Error: API 호출 실패"

def process_sheet(sheet_name):
    worksheet = authenticate_gspread(SPREADSHEET_URL, sheet_name)
    if not worksheet: return

    print(f"\n📂 [{sheet_name}] 워크시트 처리를 시작합니다...")

    data = worksheet.get_all_values()
    if not data:
        print("❌ 데이터가 없습니다.")
        return

    headers = data[0]
    df = pd.DataFrame(data[1:], columns=headers)
    input_col_idx = 1 # B열
    
    # [수정] Summary 컬럼 확인 및 할당
    if "Summary" in headers:
        summary_col_idx_0 = headers.index("Summary")
        base_result_col_start = summary_col_idx_0 + 1
    else:
        base_result_col_start = len(headers) + 1
        summary_col_idx_0 = base_result_col_start - 1
    
    # 열 2개 최소 보장 (Summary, Slide 1)
    ensure_columns(worksheet, base_result_col_start + 1)

    # 헤더 업데이트 (최초 1회, Point Form 제외)
    worksheet.update(range_name=f"{gspread.utils.rowcol_to_a1(1, base_result_col_start)}:{gspread.utils.rowcol_to_a1(1, base_result_col_start+1)}", 
                     values=[["Summary", "Thematic Grouping (Slide 1)"]])
    

    # 💡 로컬 백업용 CSV 파일 준비
    backup_file = os.path.join(BACKUP_FOLDER, f"{sheet_name}_backup.csv")
    with open(backup_file, 'a', newline='', encoding='utf-8-sig') as csvfile:
        writer = csv.writer(csvfile)
        # 백업 파일이 비어있으면 헤더 작성
        if os.stat(backup_file).st_size == 0:
            writer.writerow(["Row", "Original", "Summary", "Slide 1", "Slide 2", "..."])

    for index, row in df.iterrows():
        row_num = index + 2
        
        # [수정] 이미 요약된 내용이 있으면 스킵
        if summary_col_idx_0 < len(row):
            existing_summary = str(row.iloc[summary_col_idx_0]).strip()
            if existing_summary and existing_summary.lower() not in ['nan', 'none']:
                continue

        original_text = row.iloc[input_col_idx]

        if not original_text or pd.isna(original_text):
            continue
            
        print(f"\n🔄 [행 {row_num}] 처리 및 모니터링 중...")
        print(f"  📝 원본: {str(original_text)[:50]}...") # 진행상황 모니터링용 출력

        # API 호출 (1단계, 2단계 통합 -> Summary 1회 산출)
        summary = call_gpt(original_text, PROMPT_MAIN)

        # 그룹화 로직 (Summary의 라인 수를 기준으로 그룹화 여부 결정)
        summary_lines = [line.strip() for line in summary.split('\n') if line.strip()]
        if len(summary_lines) <= GROUPING_THRESHOLD:
            grouped_text = summary
        else:
            grouped_text = call_gpt(summary, PROMPT_GROUPING)
        
        # 분할 로직 (Chunks 생성)
        lines = [line.strip() for line in grouped_text.split('\n') if line.strip()]
        chunks, current_chunk, current_char_count = [], [], 0
        
        for line in lines:
            line_length = len(line)
            if (len(current_chunk) >= MAX_LINES_PER_CELL) or (current_char_count + line_length > MAX_CHARS_PER_CELL):
                if current_chunk: chunks.append("\n\n".join(current_chunk))
                current_chunk = [line]
                current_char_count = line_length
            else:
                current_chunk.append(line)
                current_char_count += line_length
        if current_chunk: chunks.append("\n\n".join(current_chunk))
        if not chunks: chunks = ["응답 없음"]

        # 💡 [최적화] 모아둔 결과를 1번의 통신으로 구글 시트에 행 단위(Row-level)로 쏨!
        # 기존 [res1, res2] 2개 대신 통합된 [summary] 1개만 입력합니다.
        row_results = [summary] + chunks
        
        # 열 확장 필요 시 처리
        total_cols_needed = base_result_col_start + len(row_results) - 1
        ensure_columns(worksheet, total_cols_needed)

        # 행 범위 계산 (예: C2:E2)
        start_cell = gspread.utils.rowcol_to_a1(row_num, base_result_col_start)
        end_cell = gspread.utils.rowcol_to_a1(row_num, total_cols_needed)
        
        # 구글 시트 업데이트 (1회 통신)
        try:
            worksheet.update(range_name=f"{start_cell}:{end_cell}", values=[row_results])
            print(f"  ✅ 시트 업데이트 완료! (슬라이드 {len(chunks)}장 분할)")
        except Exception as e:
            print(f"  ⚠️ 시트 업데이트 에러 (하지만 로컬에는 저장됩니다): {e}")

        # 💡 [안전 장치] 결과를 로컬 PC CSV 파일에도 실시간 한 줄씩 덧붙여 씀 (Append)
        with open(backup_file, 'a', newline='', encoding='utf-8-sig') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow([row_num, original_text] + row_results)

        # 크로스 체크할 시간 확보 및 API 호출 제한 방어용 쿨타임
        time.sleep(1.5) 

    print(f"🎉 [{sheet_name}] 워크시트 작업 끝!")

if __name__ == "__main__":
    print("🚀 주피터랩 기반 실시간 자동화 분석을 시작합니다.")
    for sheet in TARGET_SHEETS:
        process_sheet(sheet)
    print("\n✨ 모든 작업이 성공적으로 완료되었습니다!")